In [0]:

# 1. Read raw JSON from S3 (Bronze)
df_bronze = spark.read \
    .option("multiline", "true") \
    .json("s3a://aws-scraping-test-oks/raw/products/")

# 2. Clean and cast data types
df_silver = df_bronze \
    .withColumn("price", df_bronze["price"].cast("double")) \
    .withColumn("scraped_at", df_bronze["scraped_at"].cast("timestamp")) \
    .dropDuplicates(["sku", "scraped_at"])



In [0]:
# 3. Data quality checks --> Circuit Breaker
assert df_silver.filter("price IS NULL").count() == 0, "Знайдено рядки без ціни"
assert df_silver.count() == df_silver.select("sku", "scraped_at").distinct().count(), \
    "Знайдено дублікати (sku, scraped_at)"

In [0]:
# 4. Write to existing Delta table gold_db.silver_products
df_silver.write \
    .mode("overwrite") \
    .insertInto("gold_db.silver_products")

In [0]:
#%sql
#select *
#from gold_db.silver_products